# Use TensorGuard in CI / pre-commit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/05_ci_precommit.ipynb)

TensorGuard ships a CLI, a pytest plugin, a pre-commit hook, and a GitHub Action. This notebook shows the programmatic check you would wire into a CI job to fail the build on a shape regression.

In [ ]:
%pip install -q tensorguard  # on Colab; locally: pip install -e .

In [ ]:
from tensorguard import verify_architecture

def ci_gate(source, shapes):
    r = verify_architecture(source, input_shapes=shapes)
    if r.status != 'SAFE':
        raise SystemExit('TensorGuard: shape regression\n' +
                         r.bugs[0].message)
    print('TensorGuard: OK')

ci_gate('''
import torch, torch.nn as nn
class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(8, 4); self.b = nn.Linear(4, 2)
    def forward(self, x):
        return self.b(torch.relu(self.a(x)))
''', {'x': ('batch', 8)})

On the command line the same gate is:

```bash
tensorguard path/to/model.py        # exits non-zero on a bug
pytest --tensorguard                 # pytest plugin
pre-commit run tensorguard --all-files
```